## Comparación de métodos para la estimación del tiempo esperado de resolución

El problema de optimización requiere estimar el parámetro \(t_{ij}\), definido como el tiempo esperado que necesitaría el empleado \(j\) para resolver el ticket \(i\). Los modelos de regresión exploratorios aplicados previamente, como MLR, PLS y Random Forest, mostraron una capacidad predictiva limitada sobre la variable `Worked hours`. Por este motivo, antes de seleccionar el procedimiento definitivo para construir la matriz de tiempos utilizada por el optimizador, se comparan diferentes estrategias de estimación.

Se consideran cuatro métodos principales: una mediana histórica por prioridad como modelo de referencia, una estimación basada en la combinación empleado-prioridad con respaldo jerárquico, una estimación jerárquica suavizada que reduce la influencia de grupos con pocas observaciones y un modelo de efectos mixtos que incorpora explícitamente la heterogeneidad existente entre empleados. Adicionalmente, cuando la base contiene información temporal suficiente, se evalúan modelos ARIMA y Holt-Winters para determinar si la evolución histórica de los tiempos de resolución aporta capacidad predictiva adicional.

Todos los métodos se evaluarán sobre un mismo conjunto de prueba mediante RMSE, MAE, NMSE y R². El método seleccionado posteriormente será utilizado para estimar los tiempos \(t_{ij}\) de las combinaciones ticket-empleado empleadas por el COP.

In [130]:
# ============================================================
# COMPARACIÓN DE MÉTODOS PARA ESTIMAR t_ij
# ============================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from pathlib import Path

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

from sklearn.model_selection import train_test_split

import statsmodels.formula.api as smf

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.holtwinters import ExponentialSmoothing

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from pathlib import Path

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

from sklearn.model_selection import train_test_split

import statsmodels.formula.api as smf

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.holtwinters import ExponentialSmoothing

from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LinearRegression
from sklearn.cross_decomposition import PLSRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

### Configuración del análisis

En esta sección se establecen los parámetros generales utilizados para comparar los métodos de estimación. Se define la localización del archivo procedente del preprocesado en R, el tamaño del conjunto de prueba, la semilla de reproducibilidad y los parámetros que controlan la estimación jerárquica. Centralizar estos valores permite repetir posteriormente el análisis con otras bases de datos o configuraciones sin modificar el resto del código.

In [131]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [132]:
# ============================================================
# CONFIGURACIÓN
# ============================================================



CSV_PATH = Path(
    "/content/drive/MyDrive/tickets_para_optimizacion.csv"
)

RANDOM_SEED = 42

TEST_SIZE = 0.20


# Número mínimo de observaciones para considerar fiable
# una combinación empleado-prioridad.
MIN_GROUP_OBS = 5


# Parámetro de suavizado.
# Cuanto mayor sea, más se aproxima la estimación
# del empleado hacia la estimación general de la prioridad.
SHRINKAGE_LAMBDA = 10


# Frecuencia utilizada en series temporales.
# W = semanal.
TIME_FREQUENCY = "W"


# Mínimo de periodos para intentar ARIMA/Holt-Winters.
MIN_TIME_PERIODS = 20

### Lectura y reconocimiento de variables

Se carga el archivo generado durante el preprocesado en R y se identifican automáticamente las principales variables necesarias para el análisis. En particular, se requiere una variable respuesta que represente las horas trabajadas, un identificador del empleado y el nivel de prioridad del ticket. Cuando existe una variable temporal, esta se utilizará posteriormente para evaluar la conveniencia de los modelos ARIMA y Holt-Winters.

In [133]:
# ============================================================
# LECTURA DEL CSV
# ============================================================

df = pd.read_csv(
    CSV_PATH,
    keep_default_na=False
)

print("Dimensiones de la base:", df.shape)

print("\nColumnas disponibles:")

for col in df.columns:
    print("-", col)

Dimensiones de la base: (9328, 19)

Columnas disponibles:
- id_fila_original
- Issue key
- Assignee
- Team
- Issue Type
- Request Type
- Priority
- Priority level
- Worked hours
- Created-Update en Horas
- Inward Tickets
- Outward Tickets
- Inward Project
- Outward Project
- Inward
- Outward
- Total Link
- Status
- Resolution


In [134]:
# ============================================================
# IDENTIFICACIÓN DE COLUMNAS
# ============================================================

def encontrar_columna(
    dataframe,
    candidatos,
    obligatoria=True
):

    columnas_lower = {
        col.lower().strip(): col
        for col in dataframe.columns
    }

    for candidato in candidatos:

        if candidato.lower() in columnas_lower:

            return columnas_lower[
                candidato.lower()
            ]

    if obligatoria:

        raise ValueError(
            "No se encontró ninguna de estas columnas: "
            + str(candidatos)
        )

    return None


TARGET_COL = encontrar_columna(
    df,
    [
        "Worked hours",
        "Worked Hours",
        "worked_hours",
        "Hours"
    ]
)


EMPLOYEE_COL = encontrar_columna(
    df,
    [
        "Assignee",
        "Employee",
        "Empleado"
    ]
)


PRIORITY_COL = encontrar_columna(
    df,
    [
        "Priority level",
        "Priority",
        "Priority Level"
    ]
)


DATE_COL = encontrar_columna(
    df,
    [
        "Created",
        "Created date",
        "Creation date",
        "Date",
        "Fecha"
    ],
    obligatoria=False
)


print("\nVariable objetivo:", TARGET_COL)

print("Empleado:", EMPLOYEE_COL)

print("Prioridad:", PRIORITY_COL)

print("Fecha:", DATE_COL)


Variable objetivo: Worked hours
Empleado: Assignee
Prioridad: Priority level
Fecha: None


### Preparación del conjunto de modelado

Antes de construir los modelos se normalizan las variables fundamentales y se eliminan únicamente aquellas observaciones que no disponen de la información necesaria para realizar la comparación. La variable `Worked hours` se convierte a formato numérico y se conservan únicamente valores positivos. Los identificadores de empleado y prioridad se transforman a texto para garantizar una codificación consistente durante los análisis posteriores.

In [135]:
# ============================================================
# PREPARACIÓN
# ============================================================

model_df = df.copy()


model_df[TARGET_COL] = pd.to_numeric(
    model_df[TARGET_COL],
    errors="coerce"
)


model_df[EMPLOYEE_COL] = (
    model_df[EMPLOYEE_COL]
    .astype(str)
    .str.strip()
)


model_df[PRIORITY_COL] = (
    model_df[PRIORITY_COL]
    .astype(str)
    .str.strip()
    .str.upper()
)


model_df = model_df[
    model_df[TARGET_COL].notna()
    &
    (model_df[TARGET_COL] > 0)
    &
    (model_df[EMPLOYEE_COL] != "")
    &
    (model_df[PRIORITY_COL] != "")
].copy()


if DATE_COL is not None:

    model_df[DATE_COL] = pd.to_datetime(
        model_df[DATE_COL],
        errors="coerce"
    )


print(
    "Observaciones disponibles:",
    len(model_df)
)

print(
    "Empleados:",
    model_df[EMPLOYEE_COL].nunique()
)

print(
    "Prioridades:",
    model_df[PRIORITY_COL].unique()
)

Observaciones disponibles: 9326
Empleados: 40
Prioridades: ['2' '1' '3' '4']


### División en entrenamiento y prueba

Para evaluar de forma objetiva cada método se separa una parte de los datos que no será utilizada durante la estimación de sus parámetros. Cuando existe información temporal, se emplea una partición cronológica, utilizando las observaciones más antiguas para entrenamiento y las más recientes para prueba. Esta estrategia reproduce mejor un escenario operativo real, en el que el comportamiento futuro debe estimarse utilizando exclusivamente información histórica. Si no existe una variable temporal disponible, se utiliza una partición aleatoria reproducible.

In [136]:
# ============================================================
# TRAIN / TEST
# ============================================================

use_temporal_split = (
    DATE_COL is not None
    and model_df[DATE_COL].notna().sum() > 0
)


if use_temporal_split:

    model_df = (
        model_df
        .dropna(subset=[DATE_COL])
        .sort_values(DATE_COL)
        .reset_index(drop=True)
    )

    split_position = int(
        len(model_df)
        * (1 - TEST_SIZE)
    )

    train_df = (
        model_df
        .iloc[:split_position]
        .copy()
    )

    test_df = (
        model_df
        .iloc[split_position:]
        .copy()
    )

    print(
        "Se utiliza partición temporal."
    )


else:

    train_df, test_df = train_test_split(
        model_df,
        test_size=TEST_SIZE,
        random_state=RANDOM_SEED
    )

    train_df = train_df.copy()
    test_df = test_df.copy()

    print(
        "Se utiliza partición aleatoria."
    )


print(
    "\nEntrenamiento:",
    len(train_df)
)

print(
    "Prueba:",
    len(test_df)
)

Se utiliza partición aleatoria.

Entrenamiento: 7460
Prueba: 1866


### Métricas de evaluación

Todos los métodos se comparan mediante las mismas métricas. El RMSE penaliza especialmente los errores grandes, mientras que el MAE representa el error absoluto promedio en horas. El NMSE permite comparar el error del modelo con la variabilidad existente en la variable respuesta, y el coeficiente R² representa la proporción de variabilidad explicada por las estimaciones. Utilizar simultáneamente estas métricas permite evitar que la selección del modelo dependa de un único criterio.

In [137]:
# ============================================================
# FUNCIÓN DE EVALUACIÓN
# ============================================================

resultados_modelos = []


def evaluar_modelo(
    nombre,
    y_real,
    y_pred
):

    y_real = np.asarray(
        y_real,
        dtype=float
    )

    y_pred = np.asarray(
        y_pred,
        dtype=float
    )


    valid = (
        np.isfinite(y_real)
        &
        np.isfinite(y_pred)
    )


    y_real = y_real[valid]
    y_pred = y_pred[valid]


    rmse = np.sqrt(
        mean_squared_error(
            y_real,
            y_pred
        )
    )


    mae = mean_absolute_error(
        y_real,
        y_pred
    )


    variance_y = np.var(
        y_real,
        ddof=1
    )


    nmse = (
        mean_squared_error(
            y_real,
            y_pred
        )
        / variance_y
        if variance_y > 0
        else np.nan
    )


    r2 = r2_score(
        y_real,
        y_pred
    )


    resultado = {
        "Modelo": nombre,
        "N": len(y_real),
        "RMSE": rmse,
        "MAE": mae,
        "NMSE": nmse,
        "R2": r2
    }


    resultados_modelos.append(
        resultado
    )


    return resultado

### Modelo 1. Mediana histórica por prioridad

El primer modelo constituye una referencia sencilla frente a la cual comparar los enfoques posteriores. Para cada nivel de prioridad se calcula la mediana histórica de `Worked hours` utilizando exclusivamente el conjunto de entrenamiento. Cada ticket del conjunto de prueba recibe posteriormente como estimación la mediana correspondiente a su nivel de prioridad. La utilización de la mediana, en lugar de la media, reduce la sensibilidad del estimador ante valores extremos y distribuciones asimétricas.

In [138]:
# ============================================================
# MODELO 1
# MEDIANA POR PRIORIDAD
# ============================================================

global_median = (
    train_df[TARGET_COL]
    .median()
)


priority_median = (
    train_df
    .groupby(PRIORITY_COL)[TARGET_COL]
    .median()
    .to_dict()
)


pred_baseline_priority = (
    test_df[PRIORITY_COL]
    .map(priority_median)
    .fillna(global_median)
    .astype(float)
)


evaluar_modelo(
    "Baseline 1 - Mediana prioridad",
    test_df[TARGET_COL],
    pred_baseline_priority
)

{'Modelo': 'Baseline 1 - Mediana prioridad',
 'N': 1866,
 'RMSE': np.float64(35.823472032772024),
 'MAE': 20.308858967488383,
 'NMSE': np.float64(0.7071042106687728),
 'R2': 0.292516644982343}

### Modelo 2. Mediana empleado-prioridad con respaldo jerárquico

El segundo modelo incorpora información específica de cada trabajador. Para aquellas combinaciones empleado-prioridad que cuentan con un número mínimo de observaciones históricas se utiliza la mediana de las horas registradas. Cuando el histórico disponible para dicha combinación es insuficiente, la estimación retrocede al nivel inmediatamente superior, utilizando la mediana correspondiente a la prioridad del ticket y, como último respaldo, la mediana global. Este procedimiento evita realizar estimaciones poco fiables a partir de uno o dos casos históricos.

In [139]:
# ============================================================
# MODELO 2
# EMPLEADO × PRIORIDAD + FALLBACK
# ============================================================

employee_priority_stats = (
    train_df
    .groupby(
        [
            EMPLOYEE_COL,
            PRIORITY_COL
        ]
    )[TARGET_COL]
    .agg(
        median="median",
        n="size"
    )
)


def predecir_employee_priority(row):

    key = (
        row[EMPLOYEE_COL],
        row[PRIORITY_COL]
    )

    if key in employee_priority_stats.index:

        stats = employee_priority_stats.loc[
            key
        ]

        if stats["n"] >= MIN_GROUP_OBS:

            return float(
                stats["median"]
            )


    return float(
        priority_median.get(
            row[PRIORITY_COL],
            global_median
        )
    )


pred_employee_priority = (
    test_df.apply(
        predecir_employee_priority,
        axis=1
    )
)


evaluar_modelo(
    "Baseline 2 - Empleado x prioridad",
    test_df[TARGET_COL],
    pred_employee_priority
)

{'Modelo': 'Baseline 2 - Empleado x prioridad',
 'N': 1866,
 'RMSE': np.float64(34.351990057598975),
 'MAE': 18.613162587828985,
 'NMSE': np.float64(0.6502073506196676),
 'R2': 0.3494440127312066}

### Modelo 3. Estimación jerárquica suavizada

El tercer enfoque utiliza una estimación jerárquica con suavizado. En lugar de emplear directamente la mediana histórica de cada combinación empleado-prioridad, se pondera dicha estimación en función del número de observaciones disponibles. Cuando existe abundante información sobre el comportamiento de un empleado para una prioridad concreta, su comportamiento individual recibe un peso elevado. Cuando el número de observaciones es reducido, la estimación se aproxima progresivamente al comportamiento general observado para dicha prioridad. Este mecanismo reduce la inestabilidad asociada a empleados o categorías con poco histórico.

In [140]:
# ============================================================
# MODELO 3
# ESTIMACIÓN JERÁRQUICA SUAVIZADA
# ============================================================

employee_priority_full = (
    train_df
    .groupby(
        [
            EMPLOYEE_COL,
            PRIORITY_COL
        ]
    )[TARGET_COL]
    .agg(
        median="median",
        n="size"
    )
)


def predecir_shrinkage(row):

    priority = row[
        PRIORITY_COL
    ]

    employee = row[
        EMPLOYEE_COL
    ]


    priority_estimate = float(
        priority_median.get(
            priority,
            global_median
        )
    )


    key = (
        employee,
        priority
    )


    if key not in employee_priority_full.index:

        return priority_estimate


    group = employee_priority_full.loc[
        key
    ]


    n = float(
        group["n"]
    )


    employee_estimate = float(
        group["median"]
    )


    weight = (
        n
        /
        (
            n
            + SHRINKAGE_LAMBDA
        )
    )


    prediction = (
        weight
        * employee_estimate
        +
        (1 - weight)
        * priority_estimate
    )


    return prediction


pred_shrinkage = (
    test_df.apply(
        predecir_shrinkage,
        axis=1
    )
)


evaluar_modelo(
    "Modelo 3 - Jerárquico suavizado",
    test_df[TARGET_COL],
    pred_shrinkage
)

{'Modelo': 'Modelo 3 - Jerárquico suavizado',
 'N': 1866,
 'RMSE': np.float64(34.31874628255491),
 'MAE': 18.792917899359985,
 'NMSE': np.float64(0.6489494969786935),
 'R2': 0.3507025408245351}

### Modelo 4. Modelo de efectos mixtos

El cuarto método utiliza un modelo de efectos mixtos para representar simultáneamente las características generales de los tickets y las diferencias sistemáticas entre empleados. Las variables operativas del ticket se incorporan como efectos fijos, mientras que cada empleado dispone de un efecto aleatorio que representa desviaciones respecto al comportamiento medio de la organización. Este enfoque resulta especialmente adecuado para datos jerárquicos, ya que permite aprovechar la información compartida entre trabajadores y evita estimar completamente de forma independiente a aquellos empleados que cuentan con pocas observaciones históricas.

In [141]:
# ============================================================
# MODELO 4
# EFECTOS MIXTOS
# ============================================================

numeric_candidates = [
    "Inward Tickets",
    "Outward Tickets",
    "Inward Project",
    "Outward Project"
]


numeric_features = [
    col
    for col in numeric_candidates
    if col in train_df.columns
]


# Convertir cuantitativas
for col in numeric_features:

    train_df[col] = pd.to_numeric(
        train_df[col],
        errors="coerce"
    )

    test_df[col] = pd.to_numeric(
        test_df[col],
        errors="coerce"
    )


# Imputación mediante mediana del entrenamiento
for col in numeric_features:

    median_value = (
        train_df[col]
        .median()
    )

    train_df[col] = (
        train_df[col]
        .fillna(median_value)
    )

    test_df[col] = (
        test_df[col]
        .fillna(median_value)
    )


# Construcción de fórmula
fixed_terms = [
    f"C(Q('{PRIORITY_COL}'))"
]


for col in numeric_features:

    fixed_terms.append(
        f"Q('{col}')"
    )


mixed_formula = (
    f"Q('{TARGET_COL}') ~ "
    +
    " + ".join(
        fixed_terms
    )
)


print(
    "Fórmula del modelo:"
)

print(
    mixed_formula
)

Fórmula del modelo:
Q('Worked hours') ~ C(Q('Priority level')) + Q('Inward Tickets') + Q('Outward Tickets') + Q('Inward Project') + Q('Outward Project')


In [142]:
mixed_model = smf.mixedlm(
    mixed_formula,
    data=train_df,
    groups=train_df[
        EMPLOYEE_COL
    ]
)


mixed_result = mixed_model.fit(
    reml=True,
    method="lbfgs",
    maxiter=500
)


print(
    mixed_result.summary()
)

                  Mixed Linear Model Regression Results
Model:                 MixedLM    Dependent Variable:    Q('Worked hours')
No. Observations:      7460       Method:                REML             
No. Groups:            38         Scale:                 930.1801         
Min. group size:       1          Log-Likelihood:        -36100.8877      
Max. group size:       2030       Converged:             Yes              
Mean group size:       196.3                                              
--------------------------------------------------------------------------
                             Coef.  Std.Err.    z    P>|z|  [0.025  0.975]
--------------------------------------------------------------------------
Intercept                    26.857    2.943   9.126 0.000  21.089  32.626
C(Q('Priority level'))[T.2]  -5.189    1.933  -2.684 0.007  -8.977  -1.400
C(Q('Priority level'))[T.3]  95.240    2.421  39.332 0.000  90.494  99.986
C(Q('Priority level'))[T.4]  72.451   11.723

### Preparación común para los modelos predictivos

Con el objetivo de realizar una comparación homogénea entre los modelos desarrollados previamente y diferentes alternativas predictivas, se incorporan cuatro métodos adicionales: Regresión Lineal Múltiple (MLR), Partial Least Squares (PLS), Random Forest y una red neuronal de regresión. Todos los modelos se entrenan y evalúan utilizando exactamente los mismos conjuntos de entrenamiento y prueba empleados en los métodos anteriores.

Las variables predictoras incluyen el nivel de prioridad y las características cuantitativas disponibles del ticket. En Random Forest y en la red neuronal también se incorpora la identidad del empleado como variable explicativa, ya que el objetivo final consiste en estimar un tiempo específico para cada combinación ticket-empleado. Los modelos se evalúan mediante las mismas métricas RMSE, MAE, NMSE y R², permitiendo realizar una comparación directa de su capacidad para estimar `Worked hours`.

In [143]:
# ============================================================
# PREPARACIÓN COMÚN DE VARIABLES PARA MLR, PLS, RF Y NN
# ============================================================

# Variables numéricas disponibles en la base
predictor_numeric_candidates = [
    "Inward Tickets",
    "Outward Tickets",
    "Inward Project",
    "Outward Project"
]

predictor_numeric_features = [
    col
    for col in predictor_numeric_candidates
    if col in train_df.columns
]

# Convertir variables numéricas
for col in predictor_numeric_features:

    train_df[col] = pd.to_numeric(
        train_df[col],
        errors="coerce"
    )

    test_df[col] = pd.to_numeric(
        test_df[col],
        errors="coerce"
    )

    mediana = train_df[col].median()

    train_df[col] = train_df[col].fillna(mediana)
    test_df[col] = test_df[col].fillna(mediana)


# ------------------------------------------------------------
# Codificación ordinal de la prioridad
# ------------------------------------------------------------

priority_mapping = {
    "4": 1,
    "3": 2,
    "2": 3,
    "1": 4
}

train_df["Priority_numeric"] = (
    train_df[PRIORITY_COL]
    .map(priority_mapping)
)

test_df["Priority_numeric"] = (
    test_df[PRIORITY_COL]
    .map(priority_mapping)
)


# Si existiera una prioridad inesperada
priority_median_numeric = (
    train_df["Priority_numeric"]
    .median()
)

train_df["Priority_numeric"] = (
    train_df["Priority_numeric"]
    .fillna(priority_median_numeric)
)

test_df["Priority_numeric"] = (
    test_df["Priority_numeric"]
    .fillna(priority_median_numeric)
)


# Variables utilizadas por MLR y PLS
regression_features = (
    ["Priority_numeric"]
    + predictor_numeric_features
)


X_train_reg = train_df[
    regression_features
].copy()

X_test_reg = test_df[
    regression_features
].copy()

y_train_reg = train_df[
    TARGET_COL
].astype(float)

y_test_reg = test_df[
    TARGET_COL
].astype(float)


print(
    "Variables utilizadas por MLR y PLS:"
)

for variable in regression_features:
    print("-", variable)

Variables utilizadas por MLR y PLS:
- Priority_numeric
- Inward Tickets
- Outward Tickets
- Inward Project
- Outward Project


In [144]:
# ------------------------------------------------------------
# VARIABLES DE MLR Y PLS
# ------------------------------------------------------------

regression_features = (
    ["Priority_numeric"]
    + predictor_numeric_features
)

X_train_reg = train_df[
    regression_features
].copy()

X_test_reg = test_df[
    regression_features
].copy()

y_train_reg = (
    train_df[TARGET_COL]
    .astype(float)
    .copy()
)

y_test_reg = (
    test_df[TARGET_COL]
    .astype(float)
    .copy()
)


# ------------------------------------------------------------
# COMPROBACIÓN E IMPUTACIÓN FINAL DE NaN
# ------------------------------------------------------------

print("NaN antes de imputar en TRAIN:")
print(X_train_reg.isna().sum())

print("\nNaN antes de imputar en TEST:")
print(X_test_reg.isna().sum())


for col in regression_features:

    mediana_train = X_train_reg[col].median()

    X_train_reg[col] = (
        X_train_reg[col]
        .fillna(mediana_train)
    )

    X_test_reg[col] = (
        X_test_reg[col]
        .fillna(mediana_train)
    )


print("\nNaN después de imputar:")

print(
    "Train:",
    X_train_reg.isna().sum().sum()
)

print(
    "Test:",
    X_test_reg.isna().sum().sum()
)

NaN antes de imputar en TRAIN:
Priority_numeric    0
Inward Tickets      0
Outward Tickets     0
Inward Project      0
Outward Project     0
dtype: int64

NaN antes de imputar en TEST:
Priority_numeric    0
Inward Tickets      0
Outward Tickets     0
Inward Project      0
Outward Project     0
dtype: int64

NaN después de imputar:
Train: 0
Test: 0


### Modelo 5. Regresión Lineal Múltiple

La Regresión Lineal Múltiple se incorpora como modelo de referencia lineal. Su objetivo es estimar las horas de trabajo mediante una combinación lineal del nivel de prioridad y de las variables estructurales del ticket. Este modelo permite reproducir, dentro de la misma partición de entrenamiento y prueba utilizada por los demás métodos, el enfoque lineal analizado previamente durante la fase exploratoria.

In [147]:
# ============================================================
# MODELO 5
# REGRESIÓN LINEAL MÚLTIPLE
# ============================================================

mlr_model = LinearRegression()

mlr_model.fit(
    X_train_reg,
    y_train_reg
)

pred_mlr = mlr_model.predict(
    X_test_reg
)

# Evitar tiempos negativos
pred_mlr = np.maximum(
    pred_mlr,
    0.01
)


evaluar_modelo(
    "Modelo 5 - MLR",
    y_test_reg,
    pred_mlr
)


print(
    "MLR completado."
)

MLR completado.


### Modelo 6. Partial Least Squares

El modelo PLS se utiliza para representar la relación entre las variables predictoras y las horas de trabajo mediante un conjunto reducido de componentes latentes. Antes del ajuste, las variables se centran y escalan, ya que la construcción de las componentes depende de la estructura de covarianzas existente entre los predictores. El número de componentes se limita al número de variables disponibles y se establece inicialmente en dos, en coherencia con el análisis exploratorio realizado previamente.

In [148]:
# ============================================================
# MODELO 6
# PLS
# ============================================================

scaler_pls = StandardScaler()

X_train_pls = scaler_pls.fit_transform(
    X_train_reg
)

X_test_pls = scaler_pls.transform(
    X_test_reg
)


# Dos componentes, salvo que existan menos variables
n_components_pls = min(
    2,
    X_train_pls.shape[1]
)


pls_model = PLSRegression(
    n_components=n_components_pls,
    scale=False
)

pls_model.fit(
    X_train_pls,
    y_train_reg
)


pred_pls = (
    pls_model
    .predict(
        X_test_pls
    )
    .ravel()
)


pred_pls = np.maximum(
    pred_pls,
    0.01
)


evaluar_modelo(
    "Modelo 6 - PLS",
    y_test_reg,
    pred_pls
)


print(
    "PLS completado."
)

PLS completado.


### Modelo 7. Random Forest

Random Forest se incorpora como referencia no lineal y permite representar interacciones complejas entre las características del ticket y el empleado encargado de su resolución. Además de las variables cuantitativas y del nivel de prioridad, se incluye el identificador del empleado como predictor, ya que el parámetro \(t_{ij}\) debe reflejar posibles diferencias en el tiempo de resolución entre distintos recursos ante un mismo tipo de ticket.

In [149]:
# ============================================================
# MODELO 7
# RANDOM FOREST
# ============================================================

rf_numeric_features = (
    predictor_numeric_features
)

rf_categorical_features = [
    PRIORITY_COL,
    EMPLOYEE_COL
]


rf_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            rf_numeric_features
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            rf_categorical_features
        )
    ]
)


rf_model = Pipeline(
    steps=[
        (
            "preprocessor",
            rf_preprocessor
        ),
        (
            "model",
            RandomForestRegressor(
                n_estimators=300,
                min_samples_leaf=2,
                random_state=RANDOM_SEED,
                n_jobs=-1
            )
        )
    ]
)


rf_features = (
    rf_numeric_features
    + rf_categorical_features
)


rf_model.fit(
    train_df[rf_features],
    y_train_reg
)


pred_rf = rf_model.predict(
    test_df[rf_features]
)


pred_rf = np.maximum(
    pred_rf,
    0.01
)


evaluar_modelo(
    "Modelo 7 - Random Forest",
    y_test_reg,
    pred_rf
)


print(
    "Random Forest completado."
)

Random Forest completado.


### Modelo 8. Red neuronal de regresión

Finalmente, se incorpora una red neuronal multicapa con el objetivo de evaluar si una estructura no lineal más flexible permite capturar relaciones que no son adecuadamente representadas por los modelos anteriores. La red utiliza como entradas las características estructurales del ticket, el nivel de prioridad y el empleado encargado de la resolución. La arquitectura incluye dos capas ocultas con activación ReLU y una capa Dropout para reducir el riesgo de sobreajuste. La salida consiste en una única neurona destinada a estimar las horas esperadas de trabajo.

In [150]:
# ============================================================
# MODELO 8
# RED NEURONAL
# ============================================================

nn_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            predictor_numeric_features
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            [
                PRIORITY_COL,
                EMPLOYEE_COL
            ]
        )
    ]
)


X_train_nn = nn_preprocessor.fit_transform(
    train_df[
        predictor_numeric_features
        +
        [
            PRIORITY_COL,
            EMPLOYEE_COL
        ]
    ]
)


X_test_nn = nn_preprocessor.transform(
    test_df[
        predictor_numeric_features
        +
        [
            PRIORITY_COL,
            EMPLOYEE_COL
        ]
    ]
)


# Asegurar formato float
X_train_nn = np.asarray(
    X_train_nn,
    dtype=np.float32
)

X_test_nn = np.asarray(
    X_test_nn,
    dtype=np.float32
)

y_train_nn = np.asarray(
    y_train_reg,
    dtype=np.float32
)

y_test_nn = np.asarray(
    y_test_reg,
    dtype=np.float32
)


print(
    "Número de variables de entrada de la red:",
    X_train_nn.shape[1]
)

Número de variables de entrada de la red: 46


In [151]:
nn_model = Sequential([

    Dense(
        64,
        activation="relu",
        input_shape=(
            X_train_nn.shape[1],
        )
    ),

    Dropout(0.20),

    Dense(
        32,
        activation="relu"
    ),

    Dropout(0.10),

    Dense(
        1,
        activation="linear"
    )
])


nn_model.compile(
    optimizer=Adam(
        learning_rate=0.001
    ),
    loss="mse",
    metrics=["mae"]
)


early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=15,
    restore_best_weights=True
)


history_nn = nn_model.fit(
    X_train_nn,
    y_train_nn,
    validation_split=0.20,
    epochs=200,
    batch_size=32,
    callbacks=[
        early_stopping
    ],
    verbose=0
)


pred_nn = (
    nn_model
    .predict(
        X_test_nn,
        verbose=0
    )
    .ravel()
)


pred_nn = np.maximum(
    pred_nn,
    0.01
)


evaluar_modelo(
    "Modelo 8 - Red neuronal",
    y_test_nn,
    pred_nn
)


print(
    "Red neuronal completada."
)

print(
    "Épocas ejecutadas:",
    len(
        history_nn.history[
            "loss"
        ]
    )
)

Red neuronal completada.
Épocas ejecutadas: 70


### Análisis temporal previo

Los modelos ARIMA y Holt-Winters únicamente resultan apropiados cuando las observaciones presentan una estructura temporal aprovechable. Por este motivo, antes de incorporarlos a la comparación se agregan las horas de trabajo por semana y se analiza si existe suficiente histórico para construir una serie temporal. Si la base no dispone de una variable de fecha o el número de periodos es insuficiente, estos modelos se descartan automáticamente y la comparación continúa utilizando únicamente los cuatro enfoques anteriores.

In [152]:
# ============================================================
# DIAGNÓSTICO TEMPORAL
# ============================================================

TEMPORAL_MODELS_AVAILABLE = False


if DATE_COL is None:

    print(
        "No existe una columna de fecha."
    )

    print(
        "ARIMA y Holt-Winters no serán evaluados."
    )


elif not use_temporal_split:

    print(
        "No existe información temporal suficiente."
    )


else:

    temporal_train = (
        train_df
        .set_index(DATE_COL)
        .resample(TIME_FREQUENCY)[TARGET_COL]
        .median()
        .dropna()
    )


    print(
        "Número de periodos temporales:",
        len(temporal_train)
    )


    display(
        temporal_train.tail(10)
    )


    if len(temporal_train) >= MIN_TIME_PERIODS:

        TEMPORAL_MODELS_AVAILABLE = True

        print(
            "✓ Existe suficiente histórico "
            "para probar modelos temporales."
        )

    else:

        print(
            "No existe suficiente histórico "
            "para ARIMA/Holt-Winters."
        )

No existe una columna de fecha.
ARIMA y Holt-Winters no serán evaluados.


### Modelo 9. ARIMA

Como análisis complementario se evalúa un modelo ARIMA sobre la evolución semanal de la mediana de `Worked hours`. Este enfoque analiza si los tiempos de resolución presentan dependencia temporal y permite determinar hasta qué punto la evolución histórica de la carga de trabajo puede utilizarse para anticipar observaciones futuras. A diferencia de los modelos anteriores, ARIMA no incorpora directamente las características individuales del ticket o del empleado, por lo que debe interpretarse principalmente como una referencia temporal.

In [153]:
# ============================================================
# MODELO 9
# ARIMA
# ============================================================

if TEMPORAL_MODELS_AVAILABLE:

    try:

        arima_model = ARIMA(
            temporal_train,
            order=(1, 1, 1)
        )

        arima_result = (
            arima_model.fit()
        )


        # Número de semanas presentes en test
        test_weeks = (
            test_df[DATE_COL]
            .dt.to_period("W")
        )


        unique_test_weeks = (
            sorted(
                test_weeks.unique()
            )
        )


        forecast_arima = (
            arima_result.forecast(
                steps=len(
                    unique_test_weeks
                )
            )
        )


        arima_week_predictions = dict(
            zip(
                unique_test_weeks,
                forecast_arima
            )
        )


        pred_arima = (
            test_weeks
            .map(
                arima_week_predictions
            )
            .astype(float)
        )


        pred_arima = np.maximum(
            pred_arima,
            0.01
        )


        evaluar_modelo(
            "Modelo 9 - ARIMA",
            test_df[TARGET_COL],
            pred_arima
        )


    except Exception as error:

        print(
            "ARIMA no pudo estimarse:"
        )

        print(error)

### Modelo 10. Holt-Winters

Finalmente, se evalúa el método de Holt-Winters sobre la misma serie temporal semanal. Este modelo permite representar cambios en el nivel y la tendencia de los tiempos de resolución y, cuando existe suficiente histórico, incorporar patrones estacionales. Su inclusión permite comprobar si una representación explícitamente temporal proporciona estimaciones superiores a los métodos basados en información transversal de empleados y prioridades.

In [154]:
# ============================================================
# MODELO 6
# HOLT-WINTERS
# ============================================================

if TEMPORAL_MODELS_AVAILABLE:

    try:

        hw_model = ExponentialSmoothing(
            temporal_train,
            trend="add",
            seasonal=None,
            initialization_method="estimated"
        )


        hw_result = (
            hw_model.fit(
                optimized=True
            )
        )


        test_weeks = (
            test_df[DATE_COL]
            .dt.to_period("W")
        )


        unique_test_weeks = (
            sorted(
                test_weeks.unique()
            )
        )


        forecast_hw = (
            hw_result.forecast(
                len(
                    unique_test_weeks
                )
            )
        )


        hw_week_predictions = dict(
            zip(
                unique_test_weeks,
                forecast_hw
            )
        )


        pred_hw = (
            test_weeks
            .map(
                hw_week_predictions
            )
            .astype(float)
        )


        pred_hw = np.maximum(
            pred_hw,
            0.01
        )


        evaluar_modelo(
            "Modelo 10 - Holt-Winters",
            test_df[TARGET_COL],
            pred_hw
        )


    except Exception as error:

        print(
            "Holt-Winters no pudo estimarse:"
        )

        print(error)

### Comparación de resultados

Una vez obtenidas las predicciones de cada método, se comparan sus métricas de rendimiento utilizando exactamente las mismas observaciones de prueba siempre que sea posible. El RMSE se utilizará como criterio principal de ordenación, complementándolo con MAE, NMSE y R². Esta comparación permitirá seleccionar de forma empírica el procedimiento más adecuado para estimar posteriormente el parámetro \(t_{ij}\) utilizado por el modelo de optimización.

In [155]:
# ============================================================
# COMPARATIVA FINAL
# ============================================================

comparison_df = pd.DataFrame(
    resultados_modelos
)


comparison_df = (
    comparison_df
    .sort_values(
        "RMSE",
        ascending=True
    )
    .reset_index(drop=True)
)


print(
    "======================================"
)

print(
    "COMPARACIÓN FINAL DE MODELOS"
)

print(
    "======================================"
)


display(
    comparison_df.round(4)
)

COMPARACIÓN FINAL DE MODELOS


,Modelo,N,RMSE,MAE,NMSE,R2
0,Modelo 7 - Random Forest,1866,31.2602,18.8523,0.5384,0.4613
1,Modelo 8 - Red neuronal,1866,31.3879,19.0867,0.5428,0.4569
2,Modelo 3 - Jerárquico suavizado,1866,34.3187,18.7929,0.6489,0.3507
3,Baseline 2 - Empleado x prioridad,1866,34.3520,18.6132,0.6502,0.3494
4,Baseline 1 - Mediana prioridad,1866,35.8235,20.3089,0.7071,0.2925
5,Modelo 5 - MLR,1866,36.2382,23.5561,0.7236,0.2760
6,Modelo 6 - PLS,1866,36.8255,23.8030,0.7472,0.2524


In [156]:
best_model = (
    comparison_df.iloc[0]
)


print(
    "Mejor método según RMSE:"
)

print(
    best_model["Modelo"]
)


print(
    "\nRMSE:",
    round(
        best_model["RMSE"],
        4
    )
)


print(
    "MAE:",
    round(
        best_model["MAE"],
        4
    )
)


print(
    "NMSE:",
    round(
        best_model["NMSE"],
        4
    )
)


print(
    "R²:",
    round(
        best_model["R2"],
        4
    )
)

Mejor método según RMSE:
Modelo 7 - Random Forest

RMSE: 31.2602
MAE: 18.8523
NMSE: 0.5384
R²: 0.4613
